In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

import torch
import pandas as pd
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_root = Path(
    "/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl"
)

train_dir = data_root / "train_images"
test_dir = data_root / "test_images"

train_labels = pd.read_csv(data_root / "train_labels.csv")
test_df = pd.read_csv(data_root / "test.csv")

train_count = sum(1 for _ in train_dir.glob("*.png"))
test_count = sum(1 for _ in test_dir.glob("*.png"))

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Training images:", train_count)
print("Training labels:", len(train_labels))
print("Test images:", test_count)
print("Dataset ready:", train_count == 50000 and test_count == 2000)

assert torch.cuda.is_available()
assert train_count == 50000
assert test_count == 2000
assert len(train_labels) == 50000

print("Experiment 2 environment ready!")


GPU available: True
GPU: Tesla T4
Training images: 50000
Training labels: 50000
Test images: 2000
Dataset ready: True
Experiment 2 environment ready!


In [2]:

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_root = Path("/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl")
train_dir = data_root / "train_images"
train_labels = pd.read_csv(data_root / "train_labels.csv")

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.04
        )
    ], p=0.8),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.RandomErasing(
        p=0.15,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3)
    )
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

class CIFAR10CompetitionDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.image_dir / row["filename"]

        with Image.open(image_path) as image:
            image = self.transform(image.convert("RGB"))

        label = int(row["label"])

        return image, label

train_data = CIFAR10CompetitionDataset(
    train_labels,
    train_dir,
    train_transform
)

val_data = CIFAR10CompetitionDataset(
    train_labels,
    train_dir,
    val_transform
)

indices = np.random.RandomState(42).permutation(50000)

train_set = Subset(train_data, indices[:45000])
val_set = Subset(val_data, indices[45000:])

train_loader = DataLoader(
    train_set,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3,
            stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3,
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, 1,
                    stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)

        return out

class CustomResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(
            3, 64, 3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self.make_layer(64, 2, 1)
        self.layer2 = self.make_layer(128, 2, 2)
        self.layer3 = self.make_layer(256, 2, 2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)

    def make_layer(self, out_channels, blocks, stride):
        layers = [
            BasicBlock(self.in_channels, out_channels, stride)
        ]

        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(
                BasicBlock(self.in_channels, out_channels)
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

model2 = CustomResNet().to(device)

params2 = sum(
    p.numel() for p in model2.parameters() if p.requires_grad
)

assert params2 <= 5000000

x, y = next(iter(train_loader))

model2.eval()

with torch.no_grad():
    output = model2(x.to(device))

print("Device:", device)
print("Training images:", len(train_set))
print("Validation images:", len(val_set))
print("Model parameters:", params2)
print("Under 5 million:", params2 <= 5000000)
print("Batch shape:", x.shape)
print("Output shape:", output.shape)
print("Experiment 2 model setup successful!")


Device: cuda
Training images: 45000
Validation images: 5000
Model parameters: 2777674
Under 5 million: True
Batch shape: torch.Size([128, 3, 32, 32])
Output shape: torch.Size([128, 10])
Experiment 2 model setup successful!


In [3]:

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path

epochs2 = 50
learning_rate2 = 0.001

save_dir2 = Path("/kaggle/working/experiment2")
save_dir2.mkdir(parents=True, exist_ok=True)

best_path2 = save_dir2 / "best_resnet_augmented.pth"
last_path2 = save_dir2 / "last_checkpoint.pth"
history_path2 = save_dir2 / "training_history.csv"

criterion2 = nn.CrossEntropyLoss()

optimizer2 = optim.AdamW(
    model2.parameters(),
    lr=learning_rate2,
    weight_decay=0.0001
)

scheduler2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer2,
    T_max=epochs2
)

use_amp2 = device.type == "cuda"

scaler2 = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp2
)

start_epoch2 = 0
best_acc2 = 0.0
history2 = []

if last_path2.exists():
    checkpoint2 = torch.load(
        last_path2,
        map_location=device,
        weights_only=True
    )

    model2.load_state_dict(checkpoint2["model"])
    optimizer2.load_state_dict(checkpoint2["optimizer"])
    scheduler2.load_state_dict(checkpoint2["scheduler"])
    scaler2.load_state_dict(checkpoint2["scaler"])

    start_epoch2 = checkpoint2["epoch"]
    best_acc2 = checkpoint2["best_acc"]
    history2 = checkpoint2["history"]

    print("Resuming from epoch:", start_epoch2)

print("Experiment 2 started!")
print("Device:", device)
print("Model parameters:", params2)
print("Training images:", len(train_set))
print("Validation images:", len(val_set))
print("Total epochs:", epochs2, flush=True)

for epoch in range(start_epoch2, epochs2):
    model2.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer2.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            "cuda",
            enabled=use_amp2
        ):
            outputs = model2(images)
            loss = criterion2(outputs, labels)

        scaler2.scale(loss).backward()
        scaler2.step(optimizer2)
        scaler2.update()

        train_loss += loss.item() * labels.size(0)
        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()
        train_total += labels.size(0)

    model2.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(
                "cuda",
                enabled=use_amp2
            ):
                outputs = model2(images)
                loss = criterion2(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            val_total += labels.size(0)

    avg_train_loss = train_loss / train_total
    avg_val_loss = val_loss / val_total

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total

    current_lr = optimizer2.param_groups[0]["lr"]
    scheduler2.step()

    history2.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_acc": train_acc,
        "val_loss": avg_val_loss,
        "val_acc": val_acc,
        "learning_rate": current_lr
    })

    if val_acc > best_acc2:
        best_acc2 = val_acc

        torch.save({
            "model": model2.state_dict(),
            "epoch": epoch + 1,
            "val_acc": val_acc,
            "parameters": params2
        }, best_path2)

        print("New best model saved!", flush=True)

    torch.save({
        "epoch": epoch + 1,
        "model": model2.state_dict(),
        "optimizer": optimizer2.state_dict(),
        "scheduler": scheduler2.state_dict(),
        "scaler": scaler2.state_dict(),
        "best_acc": best_acc2,
        "history": history2
    }, last_path2)

    pd.DataFrame(history2).to_csv(
        history_path2,
        index=False
    )

    print(
        f"Epoch [{epoch+1}/{epochs2}] | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}% | "
        f"Best: {best_acc2:.2f}%",
        flush=True
    )

print("Experiment 2 completed!")
print("Best validation accuracy:", best_acc2)
print("Best model saved to:", best_path2)


Experiment 2 started!
Device: cuda
Model parameters: 2777674
Training images: 45000
Validation images: 5000
Total epochs: 50
New best model saved!
Epoch [1/50] | Train Loss: 1.4609 | Train Acc: 45.88% | Val Loss: 1.4571 | Val Acc: 53.40% | Best: 53.40%
New best model saved!
Epoch [2/50] | Train Loss: 1.0120 | Train Acc: 63.78% | Val Loss: 1.1322 | Val Acc: 62.50% | Best: 62.50%
New best model saved!
Epoch [3/50] | Train Loss: 0.8262 | Train Acc: 70.63% | Val Loss: 0.8572 | Val Acc: 70.38% | Best: 70.38%
New best model saved!
Epoch [4/50] | Train Loss: 0.6938 | Train Acc: 75.61% | Val Loss: 0.7752 | Val Acc: 74.98% | Best: 74.98%
Epoch [5/50] | Train Loss: 0.6033 | Train Acc: 79.04% | Val Loss: 0.9534 | Val Acc: 71.62% | Best: 74.98%
New best model saved!
Epoch [6/50] | Train Loss: 0.5401 | Train Acc: 81.26% | Val Loss: 0.6980 | Val Acc: 77.66% | Best: 77.66%
New best model saved!
Epoch [7/50] | Train Loss: 0.4958 | Train Acc: 82.65% | Val Loss: 0.5055 | Val Acc: 82.90% | Best: 82.90%
E

In [4]:

import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

data_root = Path("/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl")
test_dir = data_root / "test_images"

test_df = pd.read_csv(data_root / "test.csv")
sample_df = pd.read_csv(data_root / "sample_submission.csv")

model_path = Path("/kaggle/working/experiment2/best_resnet_augmented.pth")
output_path = Path("/kaggle/working/submission.csv")

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

class CompetitionTestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.filenames = dataframe["filename"].tolist()
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        image_path = self.image_dir / self.filenames[index]
        with Image.open(image_path) as image:
            return self.transform(image.convert("RGB"))

test_dataset = CompetitionTestDataset(
    test_df,
    test_dir,
    test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

model2 = CustomResNet().to(device)

checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=True
)

model2.load_state_dict(checkpoint["model"])
model2.eval()

predictions = []

with torch.inference_mode():
    for images in test_loader:
        images = images.to(device)
        outputs = model2(images)
        predictions.extend(outputs.argmax(1).cpu().tolist())

assert len(predictions) == 2000
assert test_df["id"].tolist() == sample_df["id"].tolist()
assert all(0 <= label <= 9 for label in predictions)

submission = sample_df[["id"]].copy()
submission["label"] = predictions

assert submission.columns.tolist() == ["id", "label"]
assert not submission.isnull().any().any()

submission.to_csv(output_path, index=False)

print("Best model epoch:", checkpoint["epoch"])
print("Validation accuracy:", checkpoint["val_acc"])
print("Total predictions:", len(predictions))
print("Submission shape:", submission.shape)
print(submission.head())
print("Submission saved:", output_path)
print("Experiment 2 prediction completed!")


Best model epoch: 49
Validation accuracy: 93.38
Total predictions: 2000
Submission shape: (2000, 2)
   id  label
0   0      6
1   1      3
2   2      3
3   3      1
4   4      3
Submission saved: /kaggle/working/submission.csv
Experiment 2 prediction completed!


In [5]:

from pathlib import Path
import zipfile

base_dir = Path("/kaggle/working")
experiment_dir = base_dir / "experiment2"
backup_path = base_dir / "Experiment2_backup.zip"

files_to_backup = [
    experiment_dir / "best_resnet_augmented.pth",
    experiment_dir / "last_checkpoint.pth",
    experiment_dir / "training_history.csv",
    base_dir / "submission.csv"
]

for file in files_to_backup:
    assert file.exists(), f"Missing file: {file}"

with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for file in files_to_backup:
        z.write(file, arcname=str(file.relative_to(base_dir)))
        print("Added:", file.name)

with zipfile.ZipFile(backup_path, "r") as z:
    assert z.testzip() is None

print("\nBackup completed!")
print("Backup file:", backup_path)
print("Size:", round(backup_path.stat().st_size / 1024**2, 2), "MB")


Added: best_resnet_augmented.pth
Added: last_checkpoint.pth
Added: training_history.csv
Added: submission.csv

Backup completed!
Backup file: /kaggle/working/Experiment2_backup.zip
Size: 39.02 MB
